# STEP 2 — 데이터 확인 (가장 중요한 단계)

머신러닝을 해보셨으면 "데이터를 먼저 본다"는 게 익숙하실 텐데,
이미지 데이터는 **볼 게 훨씬 많습니다.**

이 노트북이 알아낼 것:

1. 폴더가 어떤 축(species / camera / 유증상·무증상 / 클래스)으로 나뉘어 있는가
2. **무증상(정상) 데이터가 실제로 있는가** ← 2단계 모델을 만들 수 있는지 결정
3. JSON 라벨의 키 이름이 정확히 무엇인가
4. **개체ID로 쓸 수 있는 필드가 있는가** ← 정확도 신뢰성의 핵심
5. 병변이 이미지에서 얼마나 작은가 → ROI 크롭이 필요한가
6. 중복이 얼마나 있는가, 특히 **클래스를 넘나드는 중복**이 있는가

> 💡 스키마를 코드에 박아두지 않고 **추론**합니다. AI Hub 데이터는 버전마다
> 키 이름이 달라지는 경우가 있어서, 가정하면 거의 틀립니다.

In [ ]:
# ── 0. 환경 준비 (Colab / Kaggle 공통) ──────────────────────────
# 이 셀 하나가 리포 동기화 → 패키지 설치 → 환경 감지까지 다 합니다.
# 리포를 직접 다운로드하거나 드라이브에 올릴 필요 없습니다.
# 다시 실행하면 항상 최신 코드로 맞춰집니다 (로컬 수정은 덮어씁니다).
import os, sys, subprocess

REPO   = "https://github.com/gayeoniee/deeplearning_test.git"
BRANCH = "main"
NAME   = "deeplearning_test"
_cwd   = os.getcwd()
if os.path.basename(_cwd) == NAME and os.path.isdir(os.path.join(_cwd, ".git")):
    DIR = _cwd            # 이미 리포 안에서 재실행 중 (중첩 clone 방지)
else:
    BASE = "/content" if os.path.isdir("/content") else (
           "/kaggle/working" if os.path.isdir("/kaggle/working") else _cwd)
    DIR = os.path.join(BASE, NAME)

if os.path.isdir(os.path.join(DIR, ".git")):
    # 이미 받아둔 경우: 최신으로 강제 동기화 (shallow clone 에서도 안전)
    subprocess.run(["git", "-C", DIR, "fetch", "--depth", "1", "origin", BRANCH], check=False)
    subprocess.run(["git", "-C", DIR, "reset", "--hard", f"origin/{BRANCH}"], check=False)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, "--depth", "1", REPO, DIR], check=True)

os.chdir(DIR)
if DIR not in sys.path:
    sys.path.insert(0, DIR)

# ⚠️ 중요: 이미 import 된 src.* 는 파이썬이 캐시하고 있어서
#    파일을 갱신해도 옛날 코드가 그대로 쓰입니다. 캐시를 비웁니다.
for _m in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[_m]

print("작업 디렉터리:", os.getcwd())
print("코드 버전   :", subprocess.run(["git", "-C", DIR, "log", "--oneline", "-1"],
                                      capture_output=True, text=True).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm", "imagehash", "pyarrow", "grad-cam"], check=False)

# 한글 그래프 폰트 (Colab 기본에는 한글이 없어 □ 로 나옵니다)
_font = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(_font):
    subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False)
try:
    import matplotlib.pyplot as plt, matplotlib.font_manager as fm
    fm.fontManager.addfont(_font)
    plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False
except Exception:
    pass

from src import env
from src.config import CFG, CLASSES, CLASS_KO
E = env.describe()
env.set_seed(42)

## 1. 전체 스캔

수만 장이면 몇 분 걸립니다. 급하면 `quick=True`.

In [ ]:
from src import scan

rep = scan.run()          # quick=True 로 빠르게 훑을 수도 있습니다

## 2. 데이터셋 카드 생성

스캔 결과를 `docs/data/DATASET_CARD.md` 로 저장합니다.
나중에 "그때 데이터가 어땠지?"를 다시 확인할 수 있는 기록입니다.

In [ ]:
scan.write_dataset_card(rep)

## 3. 결과 해석 — 반드시 확인할 4가지

In [ ]:
# ① 무증상(정상) 데이터가 있는가?
print("무증상 데이터:", "있음 ✅" if rep.has_normal else "없음 ❌")
print("symptom 축:", rep.axes.get("symptom"))
print()
if rep.has_normal:
    print("→ 계획대로 2단계(정상/이상 → 병변 6종) 모델을 만들 수 있습니다.")
else:
    print("→ 음성 샘플을 따로 만들어야 합니다. 아래 3가지 중 선택:")
    print("   (a) 유증상 이미지에서 병변 '바깥' 정상 피부를 크롭   ← 1순위 추천")
    print("   (b) Stanford Dogs 등 공개 강아지 사진을 정상으로 사용 ← 촬영조건 차이 위험")
    print("   (c) 1단계를 생략하고 6종 분류 + 저신뢰 거절로 대체")

In [ ]:
# ② 개체ID를 찾았는가? — 데이터 누수를 막을 수 있는지 결정합니다
print("JSON 필드 후보:", rep.animal_id_stats.get("json_field_guess"))
print()
for c in rep.animal_id_stats.get("filename_token_candidates", []):
    print(f"  파일명 토큰 #{c['token_index']}: 고유 {c['unique']:,}개, "
          f"개체당 평균 {c['avg_per_group']}장, 예 {c['examples']}")
print()
print("→ '개체당 평균'이 2 이상이고 고유값이 충분히 많은 후보가 좋습니다.")
print("   아무것도 없으면 중복 클러스터를 그룹 대용으로 씁니다 (split.py 가 자동 처리).")

In [ ]:
# ③ 병변이 얼마나 작은가? — ROI 크롭 필요 여부
la = rep.lesion_area
if la:
    print(f"병변 면적 중앙값: {la['median']:.2%}")
    print(f"이미지의 5% 미만: {la['under_5pct']:.1%}")
    print(f"           1% 미만: {la['under_1pct']:.1%}")
    print()
    if la["under_5pct"] > 0.5:
        print("→ ROI 크롭 필수. 전체 이미지를 넣으면 모델이 배경을 학습합니다.")
    else:
        print("→ 전체 이미지도 시도해볼 만합니다. STEP 3에서 둘 다 만들어 비교하세요.")
else:
    print("좌표를 못 찾았습니다. rep.field_guess 의 polygon/bbox 후보를 확인하세요.")

In [ ]:
# ④ 중복 오염 — 선행 프로젝트들이 실패한 지점
d = rep.dup_estimate
if d:
    print(f"샘플 {d['sampled']:,}장 기준 중복률: {d['duplicate_rate']:.2%}")
    print(f"클래스를 넘나드는 중복 그룹: {d['cross_class_groups']}건")
    if d["cross_class_groups"]:
        print("\n→ 같은 사진에 서로 다른 라벨이 붙어 있습니다.")
        print("   어느 쪽이 맞는지 알 수 없으므로 STEP 3에서 전부 제거합니다.")

## 4. JSON 원본 눈으로 보기

자동 추정을 믿기 전에 실물을 한 번 보세요.

In [ ]:
import json
print(json.dumps(rep.sample_json, indent=2, ensure_ascii=False)[:2500])

In [ ]:
# 역할별 추정 결과
for role, key in rep.field_guess.items():
    print(f"  {role:11} → {key}")

## 5. 클래스 분포 시각화

In [ ]:
import matplotlib.pyplot as plt

if rep.class_counts:
    ks = list(rep.class_counts); vs = list(rep.class_counts.values())
    fig, ax = plt.subplots(figsize=(7, 3.4))
    ax.bar(ks, vs, color="#4C78A8")
    ax.set_title("클래스별 이미지 수"); ax.grid(axis="y", alpha=.3)
    for k, v in zip(ks, vs):
        ax.text(k, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
    plt.tight_layout(); plt.show()
    print(f"불균형 비 (최다/최소): {max(vs)/max(min(vs),1):.1f}배")
    print("→ 5배가 넘으면 class weight 나 weighted sampler 가 필요합니다.")

## 6. 실제 이미지 보기

**이 단계를 건너뛰지 마세요.** 숫자만 보면 절대 안 보이는 것들이 있습니다:
초점이 나갔는지, 털에 가려졌는지, 조명이 제각각인지, 사람 손이 나오는지.

In [ ]:
from pathlib import Path
import random
from PIL import Image

root = Path(rep.root)
imgs = [p for p in root.rglob("*.jpg") if "반려견" in str(p)][:5000]
random.seed(0); picks = random.sample(imgs, min(12, len(imgs)))

fig, axes = plt.subplots(3, 4, figsize=(13, 10))
for ax, p in zip(axes.flat, picks):
    ax.imshow(Image.open(p).convert("RGB")); ax.axis("off")
    ax.set_title(p.parent.name, fontsize=9)
plt.tight_layout(); plt.show()

---
## ✅ 다음 단계

**스캔 요약 출력을 그대로 복사해서 공유해주세요.** 실물 스키마에 맞춰
`labels.py` / `split.py` 를 확정한 뒤 `02_전처리_매니페스트.ipynb` 로 넘어갑니다.

📖 함께 읽기:
- [`docs/basics/02_이미지는_어떻게_숫자가_되나.md`](../docs/basics/02_이미지는_어떻게_숫자가_되나.md)
- [`docs/cautions/02_데이터_누수_가장_치명적인_함정.md`](../docs/cautions/02_데이터_누수_가장_치명적인_함정.md) ← 다음 단계 전 필독